# Analysis Pipeline Test Notebook

This notebook exercises the Python analysis logic directly, without testing audio download, MinIO access, queueing, or callbacks.

It covers:
- synthetic audio generation for a repeatable input
- transcription
- prosody feature extraction
- audio emotion scoring
- text emotion scoring
- multimodal fusion

On first execution, transformer-based models may download weights. If model loading fails in the current environment, the service fallbacks are still exercised.

In [1]:
from __future__ import annotations

import importlib
import importlib.util
import json
import math
import os
import struct
import subprocess
import sys
import tempfile
import wave
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent, cwd / 'analysis-python', cwd.parent / 'analysis-python']
PROJECT_ROOT = next((path for path in candidate_roots if (path / 'app').exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

hf_cache_dir = PROJECT_ROOT / 'tmp' / 'hf-models'
torch_cache_dir = PROJECT_ROOT / 'tmp' / 'torch-models'
hf_cache_dir.mkdir(parents=True, exist_ok=True)
torch_cache_dir.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HOME', str(hf_cache_dir))
os.environ.setdefault('TORCH_HOME', str(torch_cache_dir))
required_packages = {
    'pythonjsonlogger': 'python-json-logger',
    'librosa': 'librosa',
    'minio': 'minio',
    'sentencepiece': 'sentencepiece',
}

missing_packages = [package_name for module_name, package_name in required_packages.items() if importlib.util.find_spec(module_name) is None]

if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])

import app.services.text_emotion_model as text_emotion_model
importlib.reload(text_emotion_model)

from app.services.analysis_tasks import _fuse_emotion_scores
from app.services.prosody import classify_audio_emotions, extract_prosody_features
from app.services.text_emotion_model import analyze_text_emotions
from app.services.transcription import transcribe_audio

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/nunoaraujo/MIA/CA/CA-25_26/CA-25_26/analysis-python


In [2]:
audio_path = Path('./samples/sample_1.wav')

assert audio_path.exists(), f'Audio file not found: {audio_path}'
audio_path

PosixPath('samples/sample_1.wav')

In [3]:
transcription = transcribe_audio(str(audio_path), 'pt-BR')
print(f'Transcription: {transcription}')

Device set to use cpu
{"message": "Loaded ASR model openai/whisper-small"}
/opt/miniconda3/envs/testenv/lib/python3.13/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Transcription: Hoje tive um dia extremamente cansativo depois de passar o fim de semana com a família e em passeios, hoje tive que compensar o trabalho de um tempo perdido com muito trabalho, a tentar balancear o trabalho com universidade, com o filho foi extremamente complicado. com tudo penso que consegui cumprir a maior parte dos objetivos que tinha para o dia.


In [4]:
semantic_scores = analyze_text_emotions(transcription)
print(json.dumps(semantic_scores, indent=2, ensure_ascii=False))

Lexical scores: {'joy': 0.0, 'sadness': 0.0, 'anger': 0.0, 'anxiety': 0.0, 'calm': 0.0, 'energy': 0.0}


Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
{"message": "Loaded zero-shot text emotion model: joeddav/xlm-roberta-large-xnli"}


Zero-shot scores: {'joy': 0.0006506161880679429, 'sadness': 0.8530095815658569, 'anger': 0.7867740988731384, 'anxiety': 0.8391048908233643, 'calm': 0.0028633936308324337, 'energy': 0.003660762682557106}
{
  "joy": 0.0003578389034373686,
  "sadness": 0.46915526986122136,
  "anger": 0.43272575438022615,
  "anxiety": 0.46150768995285035,
  "calm": 0.0015748664969578387,
  "energy": 0.0020134194754064085
}


In [5]:
prosody_features = extract_prosody_features(str(audio_path))
prosody_scores = classify_audio_emotions(str(audio_path), prosody_features)

prosody_result =  {
  'prosodyScores': prosody_scores,
  'prosodyFeatures': prosody_features
}
print(json.dumps(prosody_result, indent=2, ensure_ascii=False))

/opt/miniconda3/envs/testenv/lib/python3.13/site-packages/transformers/configuration_utils.py:315: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition were not used when initializing Wav2Vec2ForSequenceClassification: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.output.bias', 'classifier.output.weight']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassific

{
  "prosodyScores": {
    "joy": 0.14119809688581314,
    "sadness": 0.2710121107266436,
    "anger": 0.12650668621063232,
    "anxiety": 0.5495171332553318,
    "calm": 0.3301931466978673,
    "energy": 0.5013199851626442
  },
  "prosodyFeatures": {
    "meanPitchHz": 111.43845943916226,
    "pitchStdDev": 18.608366397113237,
    "minPitchHz": 67.7129427339704,
    "maxPitchHz": 159.19961611844158,
    "pitchContourReg": 0.10093235170919404,
    "meanEnergy": 0.017451047897338867,
    "energyStdDev": 0.01986965909600258,
    "speechRate": 4.761904761904762,
    "pauseRatio": 0.6704152249134948,
    "mfccMean": [
      -354.5088195800781,
      56.22306442260742,
      1.706067442893982,
      21.164247512817383,
      4.607002258300781,
      -2.7846336364746094,
      -3.082044839859009,
      10.186256408691406,
      -6.2770233154296875,
      -2.6164867877960205,
      -2.575822114944458,
      -1.0713030099868774,
      -6.768170356750488
    ],
    "spectralCentroid": 2447.1581

In [6]:
emotion_vector = _fuse_emotion_scores(semantic_scores, prosody_scores)

print(json.dumps(emotion_vector, indent=2, ensure_ascii=False))

{
  "joy": 0.0426099162981501,
  "sadness": 0.40971232212084796,
  "anger": 0.340860033929348,
  "anxiety": 0.4879105229435947,
  "calm": 0.10016035055723067,
  "energy": 0.15180538918157774
}


In [7]:
def top_labels(scores: dict[str, float], limit: int = 3) -> list[tuple[str, float]]:
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]

summary = {
    'topSemantic': top_labels(semantic_scores),
    'topProsody': top_labels(prosody_scores),
    'topFused': top_labels(emotion_vector),
    'transcriptionLength': len(transcription),
}

summary

{'topSemantic': [('sadness', 0.46915526986122136),
  ('anxiety', 0.46150768995285035),
  ('anger', 0.43272575438022615)],
 'topProsody': [('anxiety', 0.5495171332553318),
  ('energy', 0.5013199851626442),
  ('calm', 0.3301931466978673)],
 'topFused': [('anxiety', 0.4879105229435947),
  ('sadness', 0.40971232212084796),
  ('anger', 0.340860033929348)],
 'transcriptionLength': 351}